# Conditional Workflow with LangGraph: Quadratic Equation Solver

### Workflow Architecture
```text
                     ┌──► [handle_real_roots] ───┐
                     │    (if D > 0)             │
                     │                           │
[START] ──► [calc_d] ───► [handle_single_root] ──┼──► [END]
                     │    (if D == 0)            │
                     │                           │
                     └──► [handle_complex_roots] ┘
                          (if D < 0)
```

In [ ]:
import math
from typing import TypedDict, Optional, Literal
from langgraph.graph import StateGraph, START, END

In [ ]:
# 1. Define State
class QuadraticState(TypedDict):
    # Input coefficients for ax^2 + bx + c = 0
    a: float
    b: float
    c: float

    # Computed discriminant: D = b^2 - 4ac
    d: Optional[float]

    # Result fields
    root_type: Optional[str]
    roots: Optional[tuple]
    result_text: Optional[str]

In [ ]:
# 2. Node: Calculate Discriminant
def calculate_discriminant(state: QuadraticState) -> dict:
    a, b, c = state["a"], state["b"], state["c"]
    if a == 0:
        raise ValueError("Coefficient 'a' cannot be 0 in a quadratic equation.")
    
    d = (b ** 2) - (4 * a * c)
    return {"d": round(d, 4)}

In [ ]:
# 3. Router Function (Conditional Edge Logic)
def route_by_discriminant(state: QuadraticState) -> Literal["real_roots", "single_root", "complex_roots"]:
    d = state["d"]
    
    if d > 0:
        return "real_roots"
    elif d == 0:
        return "single_root"
    else:
        return "complex_roots"

In [ ]:
# 4. Conditional Branch Nodes

def handle_real_roots(state: QuadraticState) -> dict:
    a, b, d = state["a"], state["b"], state["d"]
    sqrt_d = math.sqrt(d)
    
    root1 = round((-b + sqrt_d) / (2 * a), 4)
    root2 = round((-b - sqrt_d) / (2 * a), 4)
    
    return {
        "root_type": "Two Real and Distinct Roots",
        "roots": (root1, root2),
        "result_text": f"D = {d} (> 0) -> Roots are real and distinct: x1 = {root1}, x2 = {root2}"
    }


def handle_single_root(state: QuadraticState) -> dict:
    a, b, d = state["a"], state["b"], state["d"]
    root = round(-b / (2 * a), 4)
    
    return {
        "root_type": "One Real Repeated Root",
        "roots": (root,),
        "result_text": f"D = {d} (= 0) -> Root is real and repeated: x = {root}"
    }


def handle_complex_roots(state: QuadraticState) -> dict:
    a, b, d = state["a"], state["b"], state["d"]
    real_part = round(-b / (2 * a), 4)
    imag_part = round(math.sqrt(abs(d)) / (2 * a), 4)
    
    root1_str = f"{real_part} + {imag_part}i"
    root2_str = f"{real_part} - {imag_part}i"
    
    return {
        "root_type": "Two Complex Conjugate Roots",
        "roots": (root1_str, root2_str),
        "result_text": f"D = {d} (< 0) -> Roots are complex conjugates: x1 = {root1_str}, x2 = {root2_str}"
    }

In [ ]:
# 5. Build and Compile Conditional Graph
graph = StateGraph(QuadraticState)

# Add nodes
graph.add_node("calculate_discriminant", calculate_discriminant)
graph.add_node("handle_real_roots", handle_real_roots)
graph.add_node("handle_single_root", handle_single_root)
graph.add_node("handle_complex_roots", handle_complex_roots)

# Flow: START -> calculate_discriminant
graph.add_edge(START, "calculate_discriminant")

# Conditional Edge: Route based on D
graph.add_conditional_edges(
    "calculate_discriminant",
    route_by_discriminant,
    {
        "real_roots": "handle_real_roots",
        "single_root": "handle_single_root",
        "complex_roots": "handle_complex_roots"
    }
)

# All branch nodes connect to END
graph.add_edge("handle_real_roots", END)
graph.add_edge("handle_single_root", END)
graph.add_edge("handle_complex_roots", END)

# Compile Workflow
workflow = graph.compile()

In [ ]:
# 6. Test All 3 Conditional Paths

test_cases = [
    {"name": "Case 1 (D > 0): x^2 - 5x + 6 = 0", "input": {"a": 1, "b": -5, "c": 6}},
    {"name": "Case 2 (D == 0): x^2 - 4x + 4 = 0", "input": {"a": 1, "b": -4, "c": 4}},
    {"name": "Case 3 (D < 0): x^2 + 2x + 5 = 0", "input": {"a": 1, "b": 2, "c": 5}}
]

for case in test_cases:
    print(f"\n=== Testing {case['name']} ===")
    output = workflow.invoke(case["input"])
    print(f"Type: {output['root_type']}")
    print(f"Result: {output['result_text']}")

In [ ]:
# 7. Visualize the Conditional Graph
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())